In [1]:
# Above-Upper-Band short strategy (multi-symbol, Backtrader)
#
# Entry (Daily):
# - High > Upper Bollinger Band(20, 2)
# - (RSI14 - EMA9(RSI14)) is decreasing
#
# Entry (3-minute):
# - Close > VWAP
# - EMA(5) crosses below EMA(9)
#
# Exit (3-minute):
# - EMA(5) crosses above EMA(9)
# - Close of day (forced close if crossover never occurs)
#
# Stop:
# - High of day (captured as running high at entry time; no lookahead)

import sys
import subprocess
import importlib


def ensure_package(pip_name: str, import_name: str | None = None) -> None:
    import_name = import_name or pip_name
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        importlib.import_module(import_name)


ensure_package("pandas")
ensure_package("numpy")
ensure_package("backtrader")

import numpy as np
import pandas as pd
import backtrader as bt

from strategies.data import add_market_data_to_syspath, load_daily_variables

# AGENTS.md requirement: intraday data should come from price_data_import functions.
add_market_data_to_syspath()
try:
    from market_data.price_data_import import intraday_import, fragmented_intraday_import
except ImportError:
    from price_data_import import intraday_import, fragmented_intraday_import


# -----------------------------
# Parameters
# -----------------------------
DAILY_VARIABLES_PATH = None  # e.g. r"C:\\path\\to\\daily_variables.pkl"
DAILY_VARIABLES_FILENAME = "daily_variables.pkl"



LOOKBACK_DAYS = 200
END_DATE = pd.Timestamp.today().normalize()
START_DATE = END_DATE - pd.Timedelta(days=LOOKBACK_DAYS)

# intraday import settings
INTRADAY_RESAMPLE = "3min"
INTRADAY_TIMESPAN = "minute"
INTRADAY_MULTIPLIER = 1
INTRADAY_LIMIT = 50_000

# Use fragmented imports keyed to daily signal dates for better memory efficiency.
USE_FRAGMENTED_INTRADAY_IMPORT = True
GROUP_CONSECUTIVE_SIGNAL_DAYS = False

INITIAL_CASH = 100_000.0
COMMISSION = 0.001
TRADE_SIZE = 1
ALLOW_MULTIPLE_ENTRIES_PER_DAY = False


# -----------------------------
# Data loading
# -----------------------------
if DAILY_VARIABLES_PATH:
    from strategies.data import load_pickled_variables

    symbols = load_pickled_variables(DAILY_VARIABLES_PATH)
else:
    symbols = load_daily_variables(filename=DAILY_VARIABLES_FILENAME)

if not isinstance(symbols, dict):
    raise TypeError(f"Expected `symbols` to be a dict, got: {type(symbols)!r}")



Loading Variables: 100%|██████████| 26/26 [01:05<00:00,  2.51s/it]


In [2]:
# Set explicit symbols, or use AUTO_SYMBOL_COUNT from universe when TEST_SYMBOLS is None.
# TEST_SYMBOLS = ["FSLY", "CRWV", "RKLB", "CVNA", "SMR"]
AUTO_SYMBOL_COUNT = 5
TEST_SYMBOLS = list(symbols.keys())[100:500]


available_symbols = sorted(symbols.keys())
if not available_symbols:
    raise ValueError("No symbols found in loaded daily variables.")

if TEST_SYMBOLS is None:
    selected_symbols = available_symbols[: int(AUTO_SYMBOL_COUNT)]
else:
    selected_symbols = list(dict.fromkeys(TEST_SYMBOLS))

missing_symbols = [s for s in selected_symbols if s not in symbols]
if missing_symbols:
    raise KeyError(
        f"Symbols not found: {missing_symbols}. Example available symbols: {available_symbols[:20]}"
    )

print(f"Selected symbols: {selected_symbols}")
print(f"Backtest window: {START_DATE.date()} -> {END_DATE.date()} | intraday={INTRADAY_RESAMPLE}")


# -----------------------------
# Helpers
# -----------------------------
RSI_CANDIDATES = ["rsi_14", "rsi14", "rsi"]
VWAP_CANDIDATES = ["vwap", "session_vwap", "day_vwap", "intraday_vwap"]


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    return df.rename(columns={c: str(c).strip().lower().replace(" ", "_") for c in df.columns})


def ensure_datetime_index(df: pd.DataFrame, *, symbol: str, frame_name: str) -> pd.DataFrame:
    out = df.copy()

    if not isinstance(out.index, pd.DatetimeIndex):
        if "date" in out.columns:
            out["date"] = pd.to_datetime(out["date"], errors="coerce")
            out = out.set_index("date")
        elif "datetime" in out.columns:
            out["datetime"] = pd.to_datetime(out["datetime"], errors="coerce")
            out = out.set_index("datetime")
        else:
            raise TypeError(
                f"{symbol}: {frame_name} must have DatetimeIndex (or date/datetime column)."
            )

    out.index = pd.to_datetime(out.index, errors="coerce")
    out = out[~out.index.isna()].copy()

    if getattr(out.index, "tz", None) is not None:
        out.index = out.index.tz_convert(None)

    return out.sort_index()


def to_numeric(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    out = df.copy()
    for col in cols:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


def pick_first_present(df: pd.DataFrame, candidates: list[str], *, label: str, symbol: str) -> str:
    col = next((c for c in candidates if c in df.columns), None)
    if col is None:
        raise ValueError(f"{symbol}: missing {label}. Candidates: {candidates}")
    return col


def calc_rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gains = delta.clip(lower=0.0)
    losses = -delta.clip(upper=0.0)

    avg_gain = gains.ewm(alpha=1.0 / period, adjust=False, min_periods=period).mean()
    avg_loss = losses.ewm(alpha=1.0 / period, adjust=False, min_periods=period).mean()

    rs = avg_gain / avg_loss.replace(0.0, np.nan)
    rsi = 100.0 - (100.0 / (1.0 + rs))
    return rsi.fillna(100.0)


def prepare_daily_frame(symbol_data, symbol: str) -> tuple[pd.DataFrame, dict]:
    df = ensure_datetime_index(symbol_data.df, symbol=symbol, frame_name="daily")
    df = normalize_columns(df)

    if "close" not in df.columns and "adj_close" in df.columns:
        df["close"] = df["adj_close"]

    required_daily = ["open", "high", "low", "close"]
    missing = [c for c in required_daily if c not in df.columns]
    if missing:
        raise ValueError(f"{symbol}: missing required daily columns: {missing}")

    if "volume" not in df.columns:
        df["volume"] = 0.0

    df = to_numeric(df, ["open", "high", "low", "close", "volume"])
    df = df.dropna(subset=["open", "high", "low", "close"]).copy()

    rsi_col = next((c for c in RSI_CANDIDATES if c in df.columns), None)
    if rsi_col is None:
        df["rsi_14"] = calc_rsi(df["close"], period=14)
        rsi_col = "rsi_14"
    else:
        df[rsi_col] = pd.to_numeric(df[rsi_col], errors="coerce")

    # Daily signals
    rsi_14 = df[rsi_col]
    rsi_9ema = rsi_14.ewm(span=9, adjust=False, min_periods=9).mean()
    rsi_spread = rsi_14 - rsi_9ema
    spread_decreasing = rsi_spread.diff() < 0.0

    bb_mid = df["close"].rolling(window=20, min_periods=20).mean()
    bb_std = df["close"].rolling(window=20, min_periods=20).std(ddof=0)
    bb_upper = bb_mid + (2.0 * bb_std)

    high_above_upper = df["high"] > bb_upper
    daily_signal = (high_above_upper & spread_decreasing).fillna(False)

    window_mask = (df.index.normalize() >= START_DATE) & (df.index.normalize() <= END_DATE)
    signal_days = tuple(pd.Timestamp(ts).date() for ts in df.index[window_mask & daily_signal])

    daily_bt = (
        df.loc[window_mask, ["open", "high", "low", "close", "volume"]]
        .rename(
            columns={
                "open": "Open",
                "high": "High",
                "low": "Low",
                "close": "Close",
                "volume": "Volume",
            }
        )
        .dropna()
    )

    info = {
        "symbol": symbol,
        "daily_rows": int(len(daily_bt)),
        "signal_days": int(len(signal_days)),
        "latest_signal_day": signal_days[-1].isoformat() if signal_days else None,
    }

    return daily_bt, {"signal_days": signal_days, "info": info}


def prepare_intraday_frame(df_raw: pd.DataFrame, symbol: str) -> pd.DataFrame:
    if df_raw is None or df_raw.empty:
        raise ValueError(f"{symbol}: empty intraday frame")

    df = df_raw.copy()
    if not isinstance(df.index, pd.DatetimeIndex):
        if "Timestamp" in df.columns:
            df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
            df = df.set_index("Timestamp")
        elif "timestamp" in df.columns:
            df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
            df = df.set_index("timestamp")
        else:
            raise TypeError(f"{symbol}: intraday frame needs DatetimeIndex or Timestamp column")

    if getattr(df.index, "tz", None) is not None:
        df.index = df.index.tz_convert(None)

    df = normalize_columns(df)

    rename_map = {
        "open": "Open",
        "high": "High",
        "low": "Low",
        "close": "Close",
        "volume": "Volume",
    }
    for src, dst in rename_map.items():
        if src in df.columns and dst not in df.columns:
            df[dst] = df[src]

    vwap_col = pick_first_present(df, VWAP_CANDIDATES, label="VWAP column", symbol=symbol)
    if "VWAP" not in df.columns:
        df["VWAP"] = df[vwap_col]

    required = ["Open", "High", "Low", "Close", "Volume", "VWAP"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{symbol}: missing intraday columns: {missing}")

    for col in required:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    window_mask = (df.index.normalize() >= START_DATE) & (df.index.normalize() <= END_DATE)
    df = df.loc[window_mask].sort_index().dropna(subset=required)
    return df[required]


def build_fragmented_dates_dict(
    signal_days_map: dict[str, tuple],
    *,
    group_consecutive_days: bool = False,
) -> dict[str, list[tuple]]:
    """Build dates_dict for fragmented_intraday_import.

    Output format per symbol: [(from_date, to_date), ...]
    """
    out: dict[str, list[tuple]] = {}

    for sym, days in signal_days_map.items():
        unique_days = sorted({pd.Timestamp(d).date() for d in days})
        if not unique_days:
            continue

        if not group_consecutive_days:
            out[sym] = [(d, d) for d in unique_days]
            continue

        ranges: list[tuple] = []
        range_start = unique_days[0]
        range_end = unique_days[0]

        for d in unique_days[1:]:
            if (d - range_end).days <= 1:
                range_end = d
            else:
                ranges.append((range_start, range_end))
                range_start = d
                range_end = d

        ranges.append((range_start, range_end))
        out[sym] = ranges

    return out


def merge_intraday_fragments(frames: list[pd.DataFrame]) -> pd.DataFrame:
    valid_frames = [f for f in frames if isinstance(f, pd.DataFrame) and not f.empty]
    if not valid_frames:
        return pd.DataFrame()

    merged = pd.concat(valid_frames, axis=0).sort_index()
    merged = merged[~merged.index.duplicated(keep="last")]
    return merged


# -----------------------------
# Build daily context
# -----------------------------
daily_by_symbol: dict[str, pd.DataFrame] = {}
signal_days_by_symbol: dict[str, tuple] = {}
daily_scan_rows: list[dict] = []

for sym in selected_symbols:
    daily_bt, payload = prepare_daily_frame(symbols[sym], sym)
    signal_days = payload["signal_days"]

    daily_by_symbol[sym] = daily_bt
    signal_days_by_symbol[sym] = signal_days
    daily_scan_rows.append(payload["info"])

daily_scan_df = pd.DataFrame(daily_scan_rows).sort_values(["signal_days", "symbol"], ascending=[False, True])
display(daily_scan_df)

candidate_symbols = [s for s in selected_symbols if len(signal_days_by_symbol.get(s, ())) > 0]
print(f"Symbols with >=1 daily signal day in window: {len(candidate_symbols)}")

if not candidate_symbols:
    raise ValueError("No symbols passed the daily filter in the selected window.")


# -----------------------------
# Pull intraday frames
# -----------------------------
intraday_raw: dict[str, pd.DataFrame] = {}
fragment_requested_by_symbol: dict[str, int] = {}
fragment_returned_by_symbol: dict[str, int] = {}

if USE_FRAGMENTED_INTRADAY_IMPORT:
    dates_dict = build_fragmented_dates_dict(
        {sym: signal_days_by_symbol[sym] for sym in candidate_symbols},
        group_consecutive_days=bool(GROUP_CONSECUTIVE_SIGNAL_DAYS),
    )

    fragmented_raw = fragmented_intraday_import(
        dates_dict=dates_dict,
        resample=INTRADAY_RESAMPLE,
        timespan=INTRADAY_TIMESPAN,
        multiplier=INTRADAY_MULTIPLIER,
        limit=INTRADAY_LIMIT,
    )

    for sym in candidate_symbols:
        requested = len(dates_dict.get(sym, []))
        frames = fragmented_raw.get(sym, [])
        merged = merge_intraday_fragments(frames)

        fragment_requested_by_symbol[sym] = int(requested)
        fragment_returned_by_symbol[sym] = int(len(frames))
        intraday_raw[sym] = merged
else:
    intraday_raw = intraday_import(
        wl=candidate_symbols,
        from_date=START_DATE.date().isoformat(),
        to_date=END_DATE.date().isoformat(),
        resample=INTRADAY_RESAMPLE,
        timespan=INTRADAY_TIMESPAN,
        multiplier=INTRADAY_MULTIPLIER,
        limit=INTRADAY_LIMIT,
        market_open_only=True,
    )

    for sym in candidate_symbols:
        fragment_requested_by_symbol[sym] = 1
        fragment_returned_by_symbol[sym] = 1 if sym in intraday_raw else 0

intraday_by_symbol: dict[str, pd.DataFrame] = {}
run_rows: list[dict] = []

for sym in candidate_symbols:
    raw_frame = intraday_raw.get(sym)
    if raw_frame is None or raw_frame.empty:
        run_rows.append(
            {
                "symbol": sym,
                "status": "missing_intraday_data",
                "daily_signal_days": int(len(signal_days_by_symbol[sym])),
                "fragments_requested": int(fragment_requested_by_symbol.get(sym, 0)),
                "fragments_returned": int(fragment_returned_by_symbol.get(sym, 0)),
            }
        )
        continue

    try:
        intra_df = prepare_intraday_frame(raw_frame, sym)
        if intra_df.empty:
            run_rows.append(
                {
                    "symbol": sym,
                    "status": "empty_intraday_after_cleanup",
                    "daily_signal_days": int(len(signal_days_by_symbol[sym])),
                    "fragments_requested": int(fragment_requested_by_symbol.get(sym, 0)),
                    "fragments_returned": int(fragment_returned_by_symbol.get(sym, 0)),
                }
            )
            continue

        intraday_by_symbol[sym] = intra_df
        run_rows.append(
            {
                "symbol": sym,
                "status": "ok",
                "intraday_rows": int(len(intra_df)),
                "daily_signal_days": int(len(signal_days_by_symbol[sym])),
                "fragments_requested": int(fragment_requested_by_symbol.get(sym, 0)),
                "fragments_returned": int(fragment_returned_by_symbol.get(sym, 0)),
            }
        )
    except Exception as exc:
        run_rows.append(
            {
                "symbol": sym,
                "status": "error",
                "reason": str(exc),
                "daily_signal_days": int(len(signal_days_by_symbol[sym])),
                "fragments_requested": int(fragment_requested_by_symbol.get(sym, 0)),
                "fragments_returned": int(fragment_returned_by_symbol.get(sym, 0)),
            }
        )

run_prep_df = pd.DataFrame(run_rows).sort_values(["status", "symbol"]).reset_index(drop=True)
display(run_prep_df)

active_symbols = [s for s in candidate_symbols if s in intraday_by_symbol]
if not active_symbols:
    raise ValueError("No symbols left after intraday preparation.")


# -----------------------------
# Backtrader strategy
# -----------------------------
class IntradayVWAPData(bt.feeds.PandasData):
    lines = ("vwap",)
    params = (
        ("datetime", None),
        ("open", "Open"),
        ("high", "High"),
        ("low", "Low"),
        ("close", "Close"),
        ("volume", "Volume"),
        ("openinterest", -1),
        ("vwap", "VWAP"),
    )


class TradeCaptureAnalyzer(bt.Analyzer):
    def start(self) -> None:
        self.trades: list[dict] = []

    @staticmethod
    def _safe_history_value(trade, index: int, attr: str):
        try:
            return float(getattr(trade.history[index].event, attr))
        except Exception:
            return None

    def notify_trade(self, trade) -> None:
        if not trade.isclosed:
            return

        strategy = self.strategy
        symbol = trade.data._name or "UNKNOWN"

        entry_dt = bt.num2date(trade.dtopen)
        exit_dt = bt.num2date(trade.dtclose)

        self.trades.append(
            {
                "symbol": symbol,
                "entry_time": pd.Timestamp(entry_dt),
                "exit_time": pd.Timestamp(exit_dt),
                "entry_price": self._safe_history_value(trade, 0, "price"),
                "exit_price": self._safe_history_value(trade, -1, "price"),
                "size": self._safe_history_value(trade, 0, "size"),
                "pnl": float(trade.pnlcomm),
                "return_pct": (
                    (
                        self._safe_history_value(trade, 0, "price")
                        / self._safe_history_value(trade, -1, "price")
                        - 1.0
                    )
                    * 100.0
                    if self._safe_history_value(trade, -1, "price")
                    else np.nan
                ),
                "exit_reason": strategy.exit_reason_by_symbol.get(symbol),
                "entry_stop_high": strategy.last_stop_by_symbol.get(symbol),
            }
        )

    def get_analysis(self):
        return self.trades


class MultiSymbolAboveUpperBandShort(bt.Strategy):
    params = dict(
        signal_days_by_symbol=None,
        trade_size=1,
        allow_multiple_entries_per_day=False,
        # 3-minute bars: 15:57 is the standard final RTH bar.
        session_close_hour=15,
        session_close_minute=57,
        # Do not open new positions near the close.
        entry_cutoff_hour=15,
        entry_cutoff_minute=54,
    )

    def __init__(self) -> None:
        self.signal_days_by_symbol = {
            sym: {pd.Timestamp(d).date() for d in days}
            for sym, days in (self.p.signal_days_by_symbol or {}).items()
        }

        self.state = {}
        self.exit_reason_by_symbol: dict[str, str | None] = {}
        self.last_stop_by_symbol: dict[str, float | None] = {}

        for d in self.datas:
            # We keep daily feeds loaded as strict multi-timeframe context,
            # but trading logic acts on intraday feeds only.
            if str(d._name).endswith("_daily"):
                continue

            ema5 = bt.ind.EMA(d.close, period=5)
            ema9 = bt.ind.EMA(d.close, period=9)
            cross = bt.ind.CrossOver(ema5, ema9)

            self.state[d._name] = {
                "data": d,
                "cross": cross,
                "pending_order": None,
                "current_day": None,
                "running_day_high": None,
                "entry_day": None,
                "entry_stop_high": None,
                "last_entry_day": None,
                "last_day": None,
            }

    def start(self) -> None:
        for sym, st in self.state.items():
            d = st["data"]
            try:
                st["last_day"] = d.datetime.date(-1)
            except Exception:
                st["last_day"] = None
            self.exit_reason_by_symbol[sym] = None
            self.last_stop_by_symbol[sym] = None

    def notify_order(self, order) -> None:
        if order.status in [order.Submitted, order.Accepted]:
            return

        sym = order.data._name
        st = self.state.get(sym)
        if st is None:
            return

        if order.status in [order.Canceled, order.Margin, order.Rejected]:
            if order is st["pending_order"]:
                st["pending_order"] = None
            return

        if order.status != order.Completed:
            return

        if order is st["pending_order"]:
            st["pending_order"] = None

        # Entry is a sell for short strategy.
        if order.issell():
            st["entry_day"] = order.data.datetime.date(0)
            st["last_entry_day"] = st["entry_day"]
        else:
            # Exit buy-to-cover completed.
            st["entry_day"] = None
            st["entry_stop_high"] = None

    def _is_at_or_after_time(self, data, *, hour: int, minute: int) -> bool:
        dt = data.datetime.datetime(0)
        return (int(dt.hour), int(dt.minute)) >= (int(hour), int(minute))

    def _is_close_of_day_bar(self, data) -> bool:
        return self._is_at_or_after_time(
            data,
            hour=int(self.p.session_close_hour),
            minute=int(self.p.session_close_minute),
        )

    def _is_after_entry_cutoff(self, data) -> bool:
        return self._is_at_or_after_time(
            data,
            hour=int(self.p.entry_cutoff_hour),
            minute=int(self.p.entry_cutoff_minute),
        )

    def next(self) -> None:
        for sym, st in self.state.items():
            d = st["data"]

            if len(d) < 10:
                continue

            dt = d.datetime.datetime(0)
            trade_day = dt.date()

            # Track running high-of-day for stop placement.
            bar_high = float(d.high[0])
            if st["current_day"] != trade_day:
                st["current_day"] = trade_day
                st["running_day_high"] = bar_high
            else:
                st["running_day_high"] = max(float(st["running_day_high"]), bar_high)

            if st["pending_order"] is not None:
                continue

            pos = self.getposition(d)

            # Robust close-of-day exit based on 3-minute bar time.
            if pos.size < 0 and self._is_close_of_day_bar(d):
                self.exit_reason_by_symbol[sym] = "close_of_day"
                st["pending_order"] = self.close(data=d)
                continue

            # Entry block (short)
            if pos.size == 0:
                signal_days = self.signal_days_by_symbol.get(sym, set())
                if trade_day not in signal_days:
                    continue

                if (
                    (not self.p.allow_multiple_entries_per_day)
                    and (st["last_entry_day"] == trade_day)
                ):
                    continue

                # Avoid new entries into the close window.
                if self._is_after_entry_cutoff(d):
                    continue

                intraday_entry = (st["cross"][0] < 0) and (float(d.close[0]) > float(d.vwap[0]))
                if intraday_entry:
                    st["entry_stop_high"] = float(st["running_day_high"])
                    self.last_stop_by_symbol[sym] = float(st["entry_stop_high"])
                    self.exit_reason_by_symbol[sym] = None
                    st["pending_order"] = self.sell(data=d, size=int(self.p.trade_size))
                continue

            # Position management for open short.
            stop_high = st["entry_stop_high"]
            if stop_high is not None and float(d.high[0]) >= float(stop_high):
                self.exit_reason_by_symbol[sym] = "stop_high_of_day"
                st["pending_order"] = self.close(data=d)
                continue

            if st["cross"][0] > 0:
                self.exit_reason_by_symbol[sym] = "ema5_cross_above_ema9"
                st["pending_order"] = self.close(data=d)
                continue


# -----------------------------
# Run one multi-symbol Backtrader portfolio
# -----------------------------
cerebro = bt.Cerebro(stdstats=False)
cerebro.broker.setcash(INITIAL_CASH)
cerebro.broker.setcommission(commission=COMMISSION)
# Execute market orders on close for consistent bar-close signal handling.
cerebro.broker.set_coc(True)

for sym in active_symbols:
    # data0-style feeds per symbol (intraday)
    cerebro.adddata(IntradayVWAPData(dataname=intraday_by_symbol[sym]), name=sym)
    # Keep daily timeframe loaded per symbol for strict multi-timeframe setup.
    cerebro.adddata(bt.feeds.PandasData(dataname=daily_by_symbol[sym]), name=f"{sym}_daily")

cerebro.addstrategy(
    MultiSymbolAboveUpperBandShort,
    signal_days_by_symbol=signal_days_by_symbol,
    trade_size=int(TRADE_SIZE),
    allow_multiple_entries_per_day=bool(ALLOW_MULTIPLE_ENTRIES_PER_DAY),
)
cerebro.addanalyzer(TradeCaptureAnalyzer, _name="trade_capture")
cerebro.addanalyzer(bt.analyzers.TradeAnalyzer, _name="tradeanalyzer")
cerebro.addanalyzer(bt.analyzers.DrawDown, _name="drawdown")

start_value = float(cerebro.broker.getvalue())
result = cerebro.run()[0]
end_value = float(cerebro.broker.getvalue())

print(f"Symbols traded: {len(active_symbols)}")
print(f"Start value:    {start_value:,.2f}")
print(f"End value:      {end_value:,.2f}")
print(f"Net PnL:        {end_value - start_value:,.2f} ({(end_value / start_value - 1.0) * 100.0:.2f}%)")

tradeanalyzer = result.analyzers.tradeanalyzer.get_analysis()
print("TradeAnalyzer total:", tradeanalyzer.get("total"))

trades_df = pd.DataFrame(result.analyzers.trade_capture.get_analysis())
if not trades_df.empty:
    trades_df = trades_df.sort_values(["entry_time", "symbol"]).reset_index(drop=True)

    summary_df = (
        trades_df.groupby("symbol", as_index=False)
        .agg(
            trades=("pnl", "count"),
            wins=("pnl", lambda x: int((x > 0).sum())),
            total_pnl=("pnl", "sum"),
            avg_pnl=("pnl", "mean"),
            avg_return_pct=("return_pct", "mean"),
        )
        .sort_values("total_pnl", ascending=False)
        .reset_index(drop=True)
    )
    summary_df["win_rate"] = (summary_df["wins"] / summary_df["trades"]) * 100.0

    print(f"Total closed trades: {len(trades_df)}")
    display(summary_df)
    display(trades_df.head(50))
else:
    print("No closed trades were generated with current settings.")

Selected symbols: ['APLD', 'PYPL', 'SMCI', 'ANET', 'ETN', 'LMT', 'MELI', 'NET', 'TER', 'ARM', 'SLB', 'AXP', 'RDDT', 'MCD', 'BMNR', 'ACN', 'SNPS', 'COF', 'AMGN', 'SOFI', 'RBLX', 'DELL', 'F', 'KVUE', 'FITB', 'ISRG', 'NEE', 'HON', 'DDOG', 'MDT', 'B', 'UNP', 'NU', 'NKE', 'VST', 'DHR', 'SBUX', 'CVS', 'DE', 'MRVL', 'BX', 'COP', 'SAP', 'PM', 'BLK', 'CLS', 'HBAN', 'TJX', 'ALAB', 'AZN', 'LOW', 'CRDO', 'PGR', 'ONDS', 'FDX', 'TT', 'OKLO', 'IONQ', 'CDNS', 'MPWR', 'MDLZ', 'PH', 'ON', 'NXPI', 'MCHP', 'PDD', 'KMB', 'MMM', 'WELL', 'BMY', 'UPS', 'SHW', 'DASH', 'MCK', 'VRTX', 'KKR', 'CL', 'WDAY', 'PNC', 'URI', 'HWM', 'REGN', 'AG', 'MO', 'ICE', 'TGT', 'CMI', 'RCL', 'JCI', 'CRCL', 'CME', 'SYK', 'CMG', 'ADP', 'CDE', 'USB', 'ROP', 'UAL', 'FTNT', 'WBD', 'VALE', 'SATS', 'HLT', 'GM', 'EMR', 'ELV', 'EXPE', 'EQT', 'BKR', 'EQIX', 'EOG', 'AAL', 'ZTS', 'HCA', 'VLO', 'CARR', 'NOC', 'QBTS', 'CCL', 'MCO', 'DUK', 'EA', 'HL', 'AEM', 'ORLY', 'CSX', 'PAAS', 'FISV', 'MAR', 'SE', 'CB', 'TEL', 'AMT', 'WMB', 'USAR', 'AZO', 'D

,symbol,daily_rows,signal_days,latest_signal_day
122,HL,136,25,2026-01-26
88,JCI,136,24,2026-02-11
94,CDE,136,22,2026-01-26
234,CAH,136,21,2026-02-12
364,WWD,136,21,2026-02-10
...,...,...,...,...
226,KHC,136,1,2026-02-05
304,MKC,136,1,2025-11-24
42,SAP,136,1,2026-01-13
156,AJG,136,0,None


Symbols with >=1 daily signal day in window: 398


Importing Price Data: 100%|██████████| 4275/4275 [05:26<00:00, 13.08it/s]  


,symbol,status,intraday_rows,daily_signal_days,fragments_requested,fragments_returned
0,AA,ok,1631,13,13,13
1,AAL,ok,650,5,5,5
2,ABNB,ok,650,5,5,5
3,ACN,ok,910,7,7,7
4,ADM,ok,1690,13,13,13
...,...,...,...,...,...,...
393,XYZ,ok,780,6,6,6
394,YUM,ok,780,6,6,6
395,ZM,ok,780,6,6,6
396,ZS,ok,1038,8,8,8


Symbols traded: 398
Start value:    100,000.00
End value:      99,677.31
Net PnL:        -322.69 (-0.32%)
TradeAnalyzer total: AutoOrderedDict({'total': 146, 'open': 146})
No closed trades were generated with current settings.
